In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank, dense_rank, avg, when

data = [
    (1, "Ravi Kumar", "Engineering", "Hyderabad", 92000),
    (2, "Sneha Reddy", "Engineering", "Bengaluru", 97000),
    (3, "Amit Shah", "Finance", "Mumbai", 88000),
    (4, "Pooja Nair", "HR", "Chennai", 69000),
    (5, "Nikhil Verma", "Engineering", "Delhi", 85000),
    (6, "Meera Iyer", "Finance", "Hyderabad", 91000),
    (7, "Karan Malhotra", "Marketing", "Mumbai", 76000),
    (8, "Anjali Rao", "HR", "Bengaluru", 72000),
    (9, "Vivek Joshi", "Finance", "Delhi", 94000),
    (10, "Fatima Khan", "Marketing", "Chennai", 81000),
    (11, "Aditya Menon", "Engineering", "Mumbai", 99000),
    (12, "Lakshmi Das", "Finance", "Bengaluru", 87000)
]

columns = ["emp_id", "emp_name", "department", "city", "salary"]

df = spark.createDataFrame(data, columns)

display(df)



emp_id,emp_name,department,city,salary
1,Ravi Kumar,Engineering,Hyderabad,92000
2,Sneha Reddy,Engineering,Bengaluru,97000
3,Amit Shah,Finance,Mumbai,88000
4,Pooja Nair,HR,Chennai,69000
5,Nikhil Verma,Engineering,Delhi,85000
6,Meera Iyer,Finance,Hyderabad,91000
7,Karan Malhotra,Marketing,Mumbai,76000
8,Anjali Rao,HR,Bengaluru,72000
9,Vivek Joshi,Finance,Delhi,94000
10,Fatima Khan,Marketing,Chennai,81000


In [0]:

# 1. Rank employees by salary within each department
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
df = df.withColumn("rank", rank().over(window_spec))

# 2️. Create salary_level column
df = df.withColumn(
    "salary_level",
    when(col("salary") >= 95000, "Very High")
    .when(col("salary") >= 85000, "High")
    .otherwise("Standard")
)
display(df)

# 3️. Find top 2 employees from each department
top2 = df.filter(col("rank") <= 2)
display(top2)

# 4. Find average salary per department
avg_salary = df.groupBy("department") \
    .agg(avg("salary").alias("avg_salary"))
display(avg_salary)

emp_id,emp_name,department,city,salary,rank,salary_level
11,Aditya Menon,Engineering,Mumbai,99000,1,Very High
2,Sneha Reddy,Engineering,Bengaluru,97000,2,Very High
1,Ravi Kumar,Engineering,Hyderabad,92000,3,High
5,Nikhil Verma,Engineering,Delhi,85000,4,High
9,Vivek Joshi,Finance,Delhi,94000,1,High
6,Meera Iyer,Finance,Hyderabad,91000,2,High
3,Amit Shah,Finance,Mumbai,88000,3,High
12,Lakshmi Das,Finance,Bengaluru,87000,4,High
8,Anjali Rao,HR,Bengaluru,72000,1,Standard
4,Pooja Nair,HR,Chennai,69000,2,Standard


emp_id,emp_name,department,city,salary,rank,salary_level
11,Aditya Menon,Engineering,Mumbai,99000,1,Very High
2,Sneha Reddy,Engineering,Bengaluru,97000,2,Very High
9,Vivek Joshi,Finance,Delhi,94000,1,High
6,Meera Iyer,Finance,Hyderabad,91000,2,High
8,Anjali Rao,HR,Bengaluru,72000,1,Standard
4,Pooja Nair,HR,Chennai,69000,2,Standard
10,Fatima Khan,Marketing,Chennai,81000,1,Standard
7,Karan Malhotra,Marketing,Mumbai,76000,2,Standard


department,avg_salary
Engineering,93250.0
Finance,90000.0
HR,70500.0
Marketing,78500.0
